Pada Proses ini dilakukan
- penghilangan pengubahan menjadilowercase
- fix_punctuation
- penghilangan fungsi stopword_remover.remove( sastrawi )
- remove_header( Liputan 6, nama kota pada awal paragraf)
- remove_baca (menghilangkan penjelasm tulisan (baca : xxxxxx) )
- menghapus tanda baca titik yang saling bersebelahan
- membetulkan sentence domain

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#extract data ke local colabs
!tar -xzf "/content/drive/MyDrive/Colab Notebooks/Data/liputan6_data.tar.gz" -C "/content/sample_data"


In [3]:
# Load Dataset

import json
import os
import random

train_path = "/content/sample_data/liputan6_data/canonical/train"
dev_path = "/content/sample_data/liputan6_data/canonical/dev"
test_path = "/content/sample_data/liputan6_data/canonical/test"

#fungsi load urutan
def load_all_json(folder):
    data = []

    files = sorted(os.listdir(folder))[:10]

    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)

            with open(file_path, 'r') as f:
                item = json.load(f)
                data.append(item)

    return data

# fungsi load random
def load_random_json(folder, n=5000):
    data = []

    files = [f for f in os.listdir(folder) if f.endswith(".json")]

    sampled_files = random.sample(files, n)  # random tanpa duplikat

    for filename in sampled_files:
        file_path = os.path.join(folder, filename)

        with open(file_path, 'r') as f:
            item = json.load(f)
            data.append(item)

    return data

# dimatikan karena akan menggunakan random, jika ada keperluan silahkan buka
# jika dihidupkan, jangan lupa matikan random dibawah

#train_data = load_all_json(train_path)
#dev_data = load_all_json(dev_path)
#test_data = load_all_json(test_path)

train_data = load_random_json(train_path)
dev_data = load_random_json(dev_path)
test_data = load_random_json(test_path)

print("Jumlah data train_data:", len(train_data))
print("Contoh data train_data:", train_data[0])
print("--"*50)
print("Jumlah data dev_data:", len(dev_data))
print("Contoh data dev_data:", dev_data[0])
print("--"*50)
print("Jumlah data test_data:", len(test_data))
print("Contoh data test_data:", test_data[0])

Jumlah data train_data: 5000
Contoh data train_data: {'id': 198679, 'url': 'https://www.liputan6.com/news/read/198679/frustrasi--petani-bakar-lahan-pertanian', 'clean_article': [['Liputan6', '.', 'com', ',', 'Polewali', 'Mandar', ':', 'Puluhan', 'petani', 'di', 'Polewali', 'Mandar', ',', 'Sulawesi', 'Barat', ',', 'membakar', 'lahan', 'pertanian', 'mereka', '.'], ['Ini', 'dilakukan', 'sebagai', 'bentuk', 'protes', 'atas', 'kinerja', 'pemerintah', 'yang', 'dinilai', 'lamban', 'menangani', 'bencana', 'kekeringan', 'akibat', 'jebolnya', 'bendungan', 'Sekka-Sekka', 'pasacabanjir', 'bandang', 'Januari', 'lalu', '.'], ['Jebolnya', 'bendungan', 'membuat', 'distribusi', 'air', 'terhenti', '.'], ['Menurut', 'Abdul', 'Rasyid', ',', 'petani', 'setempat', ',', 'mereka', 'kesal', 'dan', 'frustrasi', 'karena', 'padinya', 'tak', 'segera', 'mendapat', 'pasokan', 'air', 'yang', 'cukup', '.'], ['Bahkan', ',', 'sebagian', 'petani', 'menyerahkan', 'lahannya', 'pada', 'peternak', 'sapi', 'untuk', 'dijadikan

In [4]:
# Convert Dataset Dan Ambil clean_artikel , clean_summary, Dan Extractive saja

def convert_to_text(data):
    new_data = []

    for item in data:
        # Gabungkan artikel
        article = " ".join(
            [" ".join(sentence) for sentence in item["clean_article"]]
        )

        # Gabungkan summary
        summary = " ".join(
            [" ".join(sentence) for sentence in item["clean_summary"]]
        )

        # 🔥 ambil  (index)
        id = item["id"]

        # 🔥 ambil extractive summary (index)
        extractive = item["extractive_summary"]

        new_data.append({
            "id": id,
            "clean_article": article,
            "clean_summary": summary,
            "extractive_summary": extractive,
        })

    return new_data

train_data_convert = convert_to_text(train_data)
dev_data_convert = convert_to_text(dev_data)
test_data_convert = convert_to_text(test_data)

In [5]:
train_data_convert[:5]

[{'id': 198679,
  'clean_article': 'Liputan6 . com , Polewali Mandar : Puluhan petani di Polewali Mandar , Sulawesi Barat , membakar lahan pertanian mereka . Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu . Jebolnya bendungan membuat distribusi air terhenti . Menurut Abdul Rasyid , petani setempat , mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup . Bahkan , sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak . Kepala Dusun Nepo , Kecamatan Wonomulyo , Logawali , menjelaskan , bendungan sudah hampir dua bulan rusak . Namun hingga kini belum ada upaya perbaikan yang berarti . Pemerintah beralasan karena tidak ada dana perbaikan . Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian kalinya . Padahal setiap kali musim tanam para petani harus mengeluarkan biaya hingga ju

In [7]:
import re

def fix_punctuation(text):
    # hapus spasi sebelum tanda baca
    text = re.sub(r'\s+([.,:;!?])', r'\1', text)

    # hapus spasi setelah tanda kurung buka
    text = re.sub(r'\(\s+', '(', text)

    # hapus spasi sebelum tanda kurung tutup
    text = re.sub(r'\s+\)', ')', text)

    # rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def remove_header(text):
    # hapus dari "liputan..." sampai ":" pertama di awal kalimat
    text = re.sub(r'^liputan\S*.*?:\s*', '', text, flags=re.IGNORECASE)
    return text

def remove_baca(text):
    text = re.sub(r'\[\s*baca.*?\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_parentheses(text):

    pattern = r'\([^)]*liputan\s?6[^)]*\)'

    cleaned_text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # rapihkan spasi
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    return cleaned_text

def multi_titik(text):
    # 1. Ubah titik dobel (.. atau . . atau .   .) jadi satu titik
    text = re.sub(r'\.\s*\.+', '.', text)

    # 2. Rapihkan spasi
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def remove_bracket_text(text):
    def replacer(match):
        content = match.group(1).strip()

        # jika isi hanya angka dan slash → anggap tanggal → jangan hapus
        if re.fullmatch(r'[\d/\-]+', content):
            return f"({content})"  # tetap dipertahankan

        return ""  # hapus

    # cari semua ( ... )
    return re.sub(r'\((.*?)\)', replacer, text)

def fix_domain(text):
    # gabungkan kembali domain yang terpisah
    text = re.sub(r'\b([A-Za-z]+)\s*\.\s*(com|co|id|net|org)\b', r'\1.\2', text)
    return text

# Penggabungan
def clean_text(text):
    # text = text.lower()
    text = fix_domain(text)
    text = fix_punctuation(text)
    # text = stopword_remover.remove(text)
    text = remove_header(text)
    text = remove_baca(text)
    text = clean_parentheses(text)
    text = remove_bracket_text(text)
    text = multi_titik(text)
    return text

In [8]:
#Apply cleaning

def apply_cleaning(data):
    new_data = []

    for item in data:
        article = clean_text(item["clean_article"])
        summary = clean_text(item["clean_summary"])

        new_data.append({
            "id": item["id"],
            "clean_article": article,
            "clean_summary": summary,
            "extractive_summary": item["extractive_summary"]
        })

    return new_data

train_text_clean = apply_cleaning(train_data_convert)
dev_text_clean = apply_cleaning(dev_data_convert)
test_text_clean = apply_cleaning(test_data_convert)

In [9]:
idx = 9  # ambil data index

print("=== ORIGINAL (convert) ===")
print(train_data_convert[idx]["clean_article"])

print("\n=== CLEANED ===")
print(train_text_clean[idx]["clean_article"])

=== ORIGINAL (convert) ===
Liputan6 . com , Jakarta : Komisi Pemilihan Umum meluncurkan pusat penghitungan suara secara elektronik di Jakarta , Sabtu ( 4/4 ) petang . Pusat tabulasi elektronik ini bertujuan agar masyarakat bisa secara cepat mengetahui hasil pemilu serta bisa mengawasi rekapitulasi penghitungan suara dari tingkat nasional hingga ke tempat pemungutan suara . Pusat tabulasi ini merupakan pusat teknologi informasi yang dibangun KPU bersama Badan Pengkajian dan Penerapan Teknologi untuk mentabulasikan jumlah suara yang masuk dari setiap kota dan kabupaten . Melalui 37 unit komputer beserta operatornya , data-data yang sudah masuk langsung dipublikasikan melalui jaringan internet yang bisa diakses langsung oleh masyarakat . Sistem penghitungan suara secara elektronik ini sebenarnya sudah tidak asing . Pada pemilu tahun 2004 , sistem ini sudah pernah digunakan . Hanya saja , karena tingkat keamanan masih lemah , data yang ada mudah dirusak orang tak bertanggung jawab . Pusat 

In [10]:
from datasets import Dataset

In [11]:
# Convert ke HuggingFace Dataset

train_text_dataset = Dataset.from_list(train_data_convert)
dev_text_dataset   = Dataset.from_list(dev_data_convert)
test_text_dataset  = Dataset.from_list(test_data_convert)

train_clean_dataset = Dataset.from_list(train_text_clean)
dev_clean_dataset   = Dataset.from_list(dev_text_clean)
test_clean_dataset  = Dataset.from_list(test_text_clean)

In [12]:
print(train_clean_dataset)
print(train_clean_dataset[0:5])

Dataset({
    features: ['id', 'clean_article', 'clean_summary', 'extractive_summary'],
    num_rows: 5000
})
{'id': [198679, 236394, 83357, 158713, 119802], 'clean_article': ['Puluhan petani di Polewali Mandar, Sulawesi Barat, membakar lahan pertanian mereka. Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu. Jebolnya bendungan membuat distribusi air terhenti. Menurut Abdul Rasyid, petani setempat, mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup. Bahkan, sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak. Kepala Dusun Nepo, Kecamatan Wonomulyo, Logawali, menjelaskan, bendungan sudah hampir dua bulan rusak. Namun hingga kini belum ada upaya perbaikan yang berarti. Pemerintah beralasan karena tidak ada dana perbaikan. Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian

In [13]:
for i in range(10):
    print("id :",train_text_dataset[i]["id"])
    print("clean_article :",train_text_dataset[i]["clean_article"])
    print("clean_summary :",train_text_dataset[i]["clean_summary"])
    print("---"*50)
    print("id :",train_clean_dataset[i]["id"])
    print("clean_article :",train_clean_dataset[i]["clean_article"])
    print("clean_summary :",train_clean_dataset[i]["clean_summary"])
    print("===="*50)

id : 198679
clean_article : Liputan6 . com , Polewali Mandar : Puluhan petani di Polewali Mandar , Sulawesi Barat , membakar lahan pertanian mereka . Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu . Jebolnya bendungan membuat distribusi air terhenti . Menurut Abdul Rasyid , petani setempat , mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup . Bahkan , sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak . Kepala Dusun Nepo , Kecamatan Wonomulyo , Logawali , menjelaskan , bendungan sudah hampir dua bulan rusak . Namun hingga kini belum ada upaya perbaikan yang berarti . Pemerintah beralasan karena tidak ada dana perbaikan . Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian kalinya . Padahal setiap kali musim tanam para petani harus mengeluarkan biaya hingga jutaan rup

In [14]:
# Simpan Ke JSON untuk keperluan upload kaggle

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_data_convert.json", "w") as f:
    json.dump(train_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_data_convert.json", "w") as f:
    json.dump(dev_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_data_convert.json", "w") as f:
    json.dump(test_data_convert, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/train_text_clean.json", "w") as f:
    json.dump(train_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/dev_text_clean.json", "w") as f:
    json.dump(dev_text_clean, f)

with open("/content/drive/MyDrive/Colab Notebooks/Data/Project02/test_text_clean.json", "w") as f:
    json.dump(test_text_clean, f)